In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
VLLM_API_KEY=os.getenv("JWT")
HF_TOKEN=os.getenv("HF_TOKEN")

In [2]:
from datasets import load_dataset
from haystack import Document

dataset = load_dataset("vblagoje/PubMedQA_instruction", split="train")
dataset = dataset.select(range(1000))
all_documents = [Document(content=doc["context"]) for doc in dataset]
all_questions = [doc["instruction"] for doc in dataset]
all_ground_truth_answers = [doc["response"] for doc in dataset]

/Users/giacomo/.venv-vllm-metal/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from haystack import Pipeline
from haystack.utils import Secret
from haystack.components.writers import DocumentWriter

In [4]:
from docling.chunking import HybridChunker
from haystack_integrations.components.converters.docling import DoclingConverter, ExportType
from haystack_integrations.document_stores.qdrant import QdrantDocumentStore
from haystack_integrations.components.embedders.vllm import VLLMDocumentEmbedder

In [5]:
EXPORT_TYPE = ExportType.DOC_CHUNKS
EMBED_MODEL_ID = "Qwen/Qwen3-Embedding-4B" # ORFEO
# EMBED_MODEL_ID = "google/embeddinggemma-300m" # DESKTOP
# EMBED_MODEL_ID = "mlx-community/Qwen3-Embedding-0.6B-8bit" # MAC
GENERATION_MODEL_ID = "Qwen/Qwen3.5-9B" # ORFEO
# GENERATION_MODEL_ID = "cyankiwi/Qwen3.5-4B-AWQ-4bit" # DESKTOP
# GENERATION_MODEL_ID = "Qwen/Qwen3-0.6B" # MAC/DESKTOP

In [6]:
indexing_pipeline = Pipeline()

document_store = QdrantDocumentStore(":memory:", embedding_dim=2560) # 768 for gemma, 1024 credo for Qwen, 2560 for Qwen3-4B

In [7]:
converter = DoclingConverter(
    export_type=EXPORT_TYPE,
    )

indexing_embedder = VLLMDocumentEmbedder(model=EMBED_MODEL_ID,
                                         api_key=Secret.from_token(VLLM_API_KEY),
                                         api_base_url="https://orfeo-llm.areasciencepark.it/vllm/v1")
                                        #  api_base_url="http://localhost:8001/v1")

writer = DocumentWriter(document_store=document_store)

In [8]:
# indexing_pipeline.add_component("converter", converter)
indexing_pipeline.add_component("embedder", indexing_embedder)
# indexing_pipeline.connect("converter", "embedder")
indexing_pipeline.add_component("writer", writer)
indexing_pipeline.connect("embedder", "writer")

🚅 Components
  - embedder: VLLMDocumentEmbedder
  - writer: DocumentWriter
🛤️ Connections
  - embedder.documents -> writer.documents (list[Document])

In [9]:
indexing_pipeline.run(
    {
        "embedder": {"documents": all_documents},
    })

Calculating embeddings: 32it [01:11,  2.25s/it]
100%|██████████| 1000/1000 [00:01<00:00, 831.53it/s]


{'embedder': {'meta': {'model': 'Qwen/Qwen3-Embedding-4B',
   'usage': {'prompt_tokens': 340081,
    'total_tokens': 340081,
    'completion_tokens': 0,
    'prompt_tokens_details': None}}},
 'writer': {'documents_written': 1000}}

Now that we have our data ready, we can create a simple RAG pipeline.

In [10]:
from haystack.components.builders import AnswerBuilder, ChatPromptBuilder
from haystack.dataclasses import ChatMessage

In [11]:
from haystack_integrations.components.retrievers.qdrant import QdrantEmbeddingRetriever

In [12]:
from haystack_integrations.components.embedders.vllm import VLLMTextEmbedder
from haystack_integrations.components.generators.vllm import VLLMChatGenerator

In [13]:
template = [
    ChatMessage.from_user(
        """
        You have to answer the following question based on the given context information only.

        Context:
        {% for document in documents %}
            {{ document.content }}
        {% endfor %}

        Question: {{question}}
        Answer:
        """
    )
]

rag_pipeline = Pipeline()

In [14]:
rag_embedder = VLLMTextEmbedder(model=EMBED_MODEL_ID,
                                api_key=Secret.from_token(VLLM_API_KEY),
                                api_base_url="https://orfeo-llm.areasciencepark.it/vllm/v1")
                                # api_base_url="http://localhost:8001/v1")

retriever = QdrantEmbeddingRetriever(document_store=document_store, top_k=3)

prompt_builder = ChatPromptBuilder(template=template)

llm = VLLMChatGenerator(model=GENERATION_MODEL_ID,
                        api_key=Secret.from_token(VLLM_API_KEY),
                        api_base_url="https://orfeo-llm.areasciencepark.it/vllm/v1")
                        # api_base_url="http://localhost:8000/v1")

answer_builder = AnswerBuilder()

ChatPromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


In [15]:
rag_pipeline.add_component("query_embedder", rag_embedder)
rag_pipeline.add_component("retriever", retriever)
rag_pipeline.connect("query_embedder.embedding", "retriever")
rag_pipeline.add_component("prompt_builder", prompt_builder)
rag_pipeline.connect("retriever", "prompt_builder")
rag_pipeline.add_component("llm", llm)
rag_pipeline.connect("prompt_builder", "llm")
rag_pipeline.add_component("answer_builder", answer_builder)
rag_pipeline.connect("llm", "answer_builder")
rag_pipeline.connect("retriever", "answer_builder")

🚅 Components
  - query_embedder: VLLMTextEmbedder
  - retriever: QdrantEmbeddingRetriever
  - prompt_builder: ChatPromptBuilder
  - llm: VLLMChatGenerator
  - answer_builder: AnswerBuilder
🛤️ Connections
  - query_embedder.embedding -> retriever.query_embedding (list[float])
  - retriever.documents -> prompt_builder.documents (list[Document])
  - retriever.documents -> answer_builder.documents (list[Document])
  - prompt_builder.prompt -> llm.messages (list[ChatMessage])
  - llm.replies -> answer_builder.replies (list[ChatMessage])

### Asking a Question

When asking a question, use the `run()` method of the pipeline. Make sure to provide the question to all components that require it as input. In this case these are the `query_embedder`, the `prompt_builder` and the `answer_builder`.

In [16]:
question = "Do high levels of procalcitonin in the early phase after pediatric liver transplantation indicate poor postoperative outcome?"

response = rag_pipeline.run(
    {
        "query_embedder": {"text": question},
        "prompt_builder": {"question": question},
        "answer_builder": {"query": question},
    }
)
print(response["answer_builder"]["answers"][0].data)



Yes, according to the context, patients with high PCT levels on postoperative day (POD) 2 suffered more often from primary graft non-function, had higher International Normalized Ratio values on POD 5, and had a longer stay in the pediatric intensive care unit and on mechanical ventilation.


## Evaluate the Pipeline

For this tutorial, let's evaluate the pipeline with the following metrics:

- [Document Mean Reciprocal Rank](https://docs.haystack.deepset.ai/docs/documentmrrevaluator): Evaluates retrieved documents using ground truth labels. It checks at what rank ground truth documents appear in the list of retrieved documents.
- [Semantic Answer Similarity](https://docs.haystack.deepset.ai/docs/sasevaluator): Evaluates predicted answers using ground truth labels. It checks the semantic similarity of a predicted answer and the ground truth answer using a fine-tuned language model.
- [Faithfulness](https://docs.haystack.deepset.ai/docs/faithfulnessevaluator): Uses an LLM to evaluate whether a generated answer can be inferred from the provided contexts. Does not require ground truth labels.


Firt, let's actually run our RAG pipeline with a set of questions, and make sure we have the ground truth labels (both answers and documents) for these questions. Let's start with 25 random questions and labels 

In [17]:
import random

questions, ground_truth_answers, ground_truth_docs = zip(
    *random.sample(list(zip(all_questions, all_ground_truth_answers, all_documents)), 50)
)

Next, let's run our pipeline and make sure to track what our pipeline returns as answers, and which documents it retrieves:

In [18]:
rag_answers = []
retrieved_docs = []

for question in list(questions):
    response = rag_pipeline.run(
        {
            "query_embedder": {"text": question},
            "prompt_builder": {"question": question},
            "answer_builder": {"query": question},
        }
    )
    print(f"Question: {question}")
    print("Answer from pipeline:")
    print(response["answer_builder"]["answers"][0].data)
    print("\n-----------------------------------\n")

    rag_answers.append(response["answer_builder"]["answers"][0].data)
    retrieved_docs.append(response["answer_builder"]["answers"][0].documents)

Question: Is high salt intake associated with a higher risk of cardiovascular events : a 7.2-year evaluation of a cohort of hypertensive patients?
Answer from pipeline:


Yes, based on the 7.2-year longitudinal retrospective evaluation of a cohort of 608 hypertensive patients, high salt intake (measured by 24-hour urinary sodium excretion, UNa) is associated with a higher risk of cardiovascular events. The study identified 24 h UNa as the only independent predictor of CV events, alongside age and night-time SBP. Specifically, a 10 mmol increase in UNa+ was associated with an HR of 1.09, and UNa+ levels above the median predicted CV events with a hazard ratio of 2.99 (95% CI, 1.75-5.13).

-----------------------------------

Question: Is self-reported body fat change in HIV-infected men a marker of decline in physical health-related quality of life with aging , independent of co-morbidity?
Answer from pipeline:


Based on the provided context, the answer is **yes**.

The longitudinal an

While each evaluator is a component that can be run individually in Haystack, they can also be added into a pipeline. This way, we can construct an `eval_pipeline` that includes all evaluators for the metrics we want to evaluate our pipeline on.

In [19]:
from haystack.components.evaluators.document_mrr import DocumentMRREvaluator
from haystack.components.evaluators.faithfulness import FaithfulnessEvaluator
from haystack.components.evaluators.sas_evaluator import SASEvaluator

eval_pipeline = Pipeline()
eval_pipeline.add_component("doc_mrr_evaluator", DocumentMRREvaluator())
eval_pipeline.add_component("faithfulness", FaithfulnessEvaluator(chat_generator=VLLMChatGenerator(model="google/gemma-4-26B-A4B-it", 
                                                                                                   api_key=Secret.from_token(VLLM_API_KEY),
                                                                                                   generation_kwargs={"response_format": {"type": "json_object"}}, 
                                                                                                   api_base_url="https://orfeo-llm.areasciencepark.it/vllm/v1")))
                                                                                                #    api_base_url="http://localhost:8000/v1")))
eval_pipeline.add_component("sas_evaluator", SASEvaluator(model="sentence-transformers/all-MiniLM-L6-v2"))

results = eval_pipeline.run(
    {
        "doc_mrr_evaluator": {
            "ground_truth_documents": list([d] for d in ground_truth_docs),
            "retrieved_documents": retrieved_docs,
        },
        "faithfulness": {
            "questions": list(questions),
            "contexts": list([d.content] for d in ground_truth_docs),
            "predicted_answers": rag_answers,
        },
        "sas_evaluator": {"predicted_answers": rag_answers, "ground_truth_answers": list(ground_truth_answers)},
    }
)

PromptBuilder has 3 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.
100%|██████████| 50/50 [00:42<00:00,  1.16it/s]
Component 'faithfulness' (type: FaithfulnessEvaluator) returned output keys {'meta'} that are not declared in its output types. These keys will be ignored and not passed to downstream components. Make sure the component's output keys match its declared @component.output_types.


### Constructing an Evaluation Report

Once we've run our evaluation pipeline, we can also create a full evaluation report. Haystack provides an `EvaluationRunResult` which we can use to display an `aggregated_report` 👇

In [20]:
from haystack.evaluation.eval_run_result import EvaluationRunResult

inputs = {
    "question": list(questions),
    "contexts": list([d.content] for d in ground_truth_docs),
    "answer": list(ground_truth_answers),
    "predicted_answer": rag_answers,
}

evaluation_result = EvaluationRunResult(run_name="pubmed_rag_pipeline", inputs=inputs, results=results)
evaluation_result.aggregated_report()

{'metrics': ['doc_mrr_evaluator', 'faithfulness', 'sas_evaluator'],
 'score': [1.0,
  np.float64(0.9803333333333333),
  np.float64(0.6981454032659531)]}

#### Extra: You can also see a detailed report with the scores for each sample in your dataset, and we will choose the output format as DataFrame

In [21]:
results_df = evaluation_result.detailed_report(output_format="df")
results_df

,question,contexts,answer,predicted_answer,doc_mrr_evaluator,faithfulness,sas_evaluator
0,Is high salt intake associated with a higher r...,[It is controversial whether high salt intake ...,"In a cohort of hypertensive patients, high sal...","\n\nYes, based on the 7.2-year longitudinal re...",1.0,1.000000,0.758834
1,Is self-reported body fat change in HIV-infect...,[Self-perception of changes in body fat among ...,Negative self-perceived body fat changes were ...,"\n\nBased on the provided context, the answer ...",1.0,1.000000,0.689283
2,Are atherogenic index of plasma and atherogeni...,[There is a robust comorbidity between mood di...,The findings suggest that lipid abnormalities ...,"\n\nYes, based on the provided context, the at...",1.0,1.000000,0.729205
3,Does comprehensive analysis suggest simple pro...,[Current basic or more advanced methods for an...,We show how these processes accord with and pr...,"\n\nBased on the provided context, the compreh...",1.0,1.000000,0.299143
4,Does mEF2D-BCL9 Fusion Gene be Associated With...,[Acute lymphoblastic leukemia (ALL) makes up a...,A novel MEF2D-BCL9 fusion we identified charac...,"\n\nYes, based on the context, a positive asso...",1.0,1.000000,0.575639
5,Does the T-box transcription factor Brachyury ...,[Epithelial-to-mesenchymal transition (EMT) in...,Our results demonstrate for the first time tha...,"\n\nYes, based on the provided context, the T-...",1.0,1.000000,0.799788
6,Does dietary beet pulp decrease taurine status...,[It is known that large dogs who are fed lamb ...,These results do not support the hypothesis th...,"\n\nBased on the provided context, the study i...",1.0,1.000000,0.666078
7,Are high Atopobium vaginae and Gardnerella vag...,[Bacterial vaginosis is a risk factor for pret...,High vaginal loads of A. vaginae and G. vagina...,"\n\nYes, according to the context, high vagina...",1.0,1.000000,0.627971
8,Do trends in outpatient MRI seem to reflect re...,[To determine whether recent reimbursement cut...,Although the majority of Medicare outpatient M...,"\n\nBased on the context, a study was conducte...",1.0,1.000000,0.800587
9,Do [ Clinical significance of continuous throm...,[To explore the relationship between continuou...,Duration of thrombocytopenia within PBD 10 is ...,"\n\nYes, based on the provided context, contin...",1.0,1.000000,0.836194


Having our evaluation results as a dataframe can be quite useful. For example, below we can use the pandas dataframe to filter the results to the top 3 best scores for semantic answer similarity (`sas_evaluator`) as well as the bottom 3 👇


In [22]:
import pandas as pd

top_3 = results_df.nlargest(3, "sas_evaluator")
bottom_3 = results_df.nsmallest(3, "sas_evaluator")
pd.concat([top_3, bottom_3])

,question,contexts,answer,predicted_answer,doc_mrr_evaluator,faithfulness,sas_evaluator
34,Does double-Stranded RNA interact With Toll-Li...,[Lung contusion is a major risk factor for the...,We conclude that the interaction of double-str...,"\n\nBased on the context provided, the binding...",1.0,1.00,0.933150
44,Does mR imaging relaxometry allow noninvasive ...,[To demonstrate the feasibility of in vivo mon...,MR imaging relaxation and diffusion measuremen...,"\n\nYes. According to the context, the study d...",1.0,1.00,0.902018
40,"Is which better for gastric cancer patients , ...",[The preferred chemotherapy method for gastric...,Perioperative chemotherapy showed improved sur...,"\n\nBased on the provided context, **periopera...",1.0,1.00,0.892551
3,Does comprehensive analysis suggest simple pro...,[Current basic or more advanced methods for an...,We show how these processes accord with and pr...,"\n\nBased on the provided context, the compreh...",1.0,1.00,0.299143
20,Is p63 more sensitive and specific than 34βE12...,[Prostate cancer is the world's leading cause ...,Basal cell markers can help to distinguish pro...,"\n\nBased on the provided context, there is no...",1.0,1.00,0.447309
39,Does poor sleep predict symptoms of depression...,[Disturbed sleep is associated with mood disor...,Lack of baseline diagnostic interviews; sleep ...,"\n\nYes, according to the provided context, po...",1.0,0.75,0.496504


## DeepEval